In [4]:
# transferable code functions
import numpy as np
import scipy.signal as scisig


def padding(layer:np.ndarray, mode:str = 'zero', pad_size:int = 1):

    for i in range(pad_size):

        padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
        padded_ly = np.zeros(padded_ly_size)
        padded_ly[1:-1,1:-1] = layer
    
        if mode == 'same':
            padded_ly[0,1:-1] = layer[0,:]
            padded_ly[-1,1:-1] = layer[-1,:]

            padded_ly[1:-1,0] = layer[:,0]
            padded_ly[1:-1,-1] = layer[:,-1]

            padded_ly[0,0] = layer[0,0]
            padded_ly[0,-1] = layer[0,-1]
            padded_ly[-1,0] = layer[-1,0]
            padded_ly[-1,-1] = layer[-1,-1]
        layer = padded_ly

    return(padded_ly)

def convolve(input:np.ndarray, kernel:np.ndarray, pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = kernel.shape[0]//2
        input = padding(input, pad_mode, pad_size)
    k_shp = kernel.shape
    #print(input.shape)
    result = scisig.convolve(input, kernel, mode=conv_mode, method="fft")

    return(result)

def patch_convolve(input:np.ndarray, patch_shape:tuple, error_array:np.ndarray, pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = patch_shape[0]//2
        input = padding(input, pad_mode, pad_size)
    
    error_patch_arr = np.zeros(shape=(*error_array.shape, *patch_shape))

    for i in range(error_array.shape[0]):
        for j in range(error_array.shape[1]):
            error_patch_arr[i,j,:,:] = input[i:i+patch_shape[0], j:j+patch_shape[1]] * error_array[i,j]
    kernel_delta = np.sum(error_patch_arr, axis=(0,1))
    return(kernel_delta)

def fast_patch_convolve(input:np.ndarray, patch_shape:tuple, error_array:np.ndarray, 
                        pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    from numpy.lib.stride_tricks import sliding_window_view
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = patch_shape[0]//2
        input = padding(input, pad_mode, pad_size)

    patches = sliding_window_view(input, (3,3))
    op1 = patches * error_array[:,:, np.newaxis, np.newaxis] #Need to understand this better

    dW = np.sum(op1, axis=(0,1))
    #dw is the delta in the filter/kernel/patch
    return(dW)

def pool(input:np.ndarray, stride = 2, mode = "max"):
    shape = (int(input.shape[0]/stride), int(input.shape[1]/stride))
    output = np.zeros(shape)
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            if mode =="max":
                output[i,j] = np.max(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
                #look at that absolute index fuckery
            if mode =="mean":
                output[i,j] = np.mean(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
    return(output)

def sigmond(input):
    return( 1/(1+np.exp(-input)))

def sigmond_prime(input):
    return(sigmond(input) * (1-sigmond(input)))

def ReLU(input):
    return(np.maximum(0, input))

def ReLU_prime(input):
    return(input > 0).astype(input.dtype)



In [ ]:
import numpy as np
from omegaconf import DictConfig, OmegaConf

def label_to_output(int) -> np.ndarray:
    outlayer = np.zeros(shape=(10), dtype=np.float32)
    outlayer[int] = 1
    return(outlayer)

def label_vec_to_output(inputs: np.ndarray) -> np.ndarray:
    # Create an identity matrix of size 10 and index into it
    return np.eye(10, dtype=np.int8)[inputs]

def create_batch(trn_cfg, image_file, label_file):
    """
    Given a hyper parameter cfg object, a loaded image file, and label file, 
    reads from both files and returns:

    input_batch: an ndarray of shape (batch_size, image_size_height, image_size_width) of type float32
    
    label_batch: an ndarray of shape (batch_size, possible_labels) of type int64
    """
    image_buffer = image_file.read(trn_cfg.image_size[0] * trn_cfg.image_size[1] * trn_cfg.batch_size)
    image_batch = np.frombuffer(image_buffer, dtype=np.uint8).astype(np.float32)
    image_batch = np.reshape(image_batch, shape = (trn_cfg.batch_size, 
                                                    trn_cfg.image_size[0], 
                                                    trn_cfg.image_size[1]))
    label_buffer = label_file.read(trn_cfg.batch_size)
    label_pre_batch = np.frombuffer(label_buffer, dtype=np.uint8).astype(np.int64)
    label_batch = label_vec_to_output(label_pre_batch)

    return(image_batch, label_batch)



In [16]:
from omegaconf import DictConfig, OmegaConf
lrn_cfg = OmegaConf.load("hyper_params.yaml")

f = open(lrn_cfg.images_path, 'rb')
f.read(16)
l = open(lrn_cfg.labels_path,'rb')
l.read(8)

image_batch, label_pre_batch = create_batch(lrn_cfg, f, l)
#print(image_batch[1,:,:], image_batch.shape)
#print(label_batch[1,:], label_batch.shape)
lrn_cfg.epochs = 1000
print(lrn_cfg.epochs)
lrn_cfg = OmegaConf.load("hyper_params.yaml")
print(lrn_cfg.epochs)


1000
10


In [ ]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import hydra
from omegaconf import DictConfig, OmegaConf
mdl_cfg = OmegaConf.load("model.yaml")
import numpy as np
from dataclasses import dataclass
from omegaconf import DictConfig
import scipy.signal as scisig


def build_index_lookup(cfg: DictConfig):
    """
    Given a loaded OmegaConf config with a 'layers' section,
    build a lookup table: index -> (layer_name, layer_data).
    """
    blocks = cfg.blocks

    index_lookup = {
        block_data.index: (block_name, block_data)
        for block_name, block_data in blocks.items()
    }
    return index_lookup


class Input_block:
    def __init__(self, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "input"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)

    def backprop(self, expected:np.ndarray=None):
        return(0)

class FC_block:
    def __init__(self, prev_ly_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_ly_shape = prev_ly_shape

        self.b_type = "fc"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.W_delta = np.zeros_like(self.weights)        
        self.B_delta = np.zeros_like(self.biases)
        self.fc_weights_shape = self.weights.shape


    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_ly_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_ly_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        if len(input.shape) == 1:
            self.z_values = np.tensordot(input, self.weights, axes=((0),(0)))  + self.biases
        if len(input.shape) == 2:
            self.z_values = np.tensordot(input, self.weights, axes=((0,1),(0,1)))  + self.biases
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if expected is not None:
            del_l = (expected - self.activations) * ReLU_prime(self.z_values)

        #print("weights shape: ", self.weights.shape)
        #print("current layer shape: ", self.layer_shape)
        #print("previous layer shape: ", self.prev_ly_shape)
        #print("del_l shape: ", del_l.shape)
        if len(self.prev_ly_shape) == 1:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((1),(0)))# * prior_z_vals
        if len(self.prev_ly_shape) == 2:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((2),(0)))# * prior_z_vals

        self.W_delta = np.tensordot(prior_activations, del_l, axes=0)
        self.B_delta = del_l

        #print("\nweights shape: ", self.weights.shape)
        #print("weights Delta shape: ", self.W_delta.shape)
        return(del_l_neg1, self.W_delta, self.B_delta)

class FC_CONV_block:
    def __init__(self, prev_bk_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_shape = prev_bk_shape

        self.b_type = "fc_conv"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.fc_weights_shape = self.weights.shape

    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_bk_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_bk_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        self.z_values = np.tensordot(input, self.weights, axes=((0,1,2),(0,1,2)))
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if expected is not None:
            del_l = (expected - self.activations) * ReLU_prime(self.z_values)

        #print("weights shape: ", self.weights.shape)
        #print("current layer shape: ", self.layer_shape)
        #print("previous layer shape: ", self.prev_bk_shape)
        #print("del_l shape: ", del_l.shape)
        
        if len(self.layer_shape) == 2:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((3,4),(0,1)))# * prior_z_vals
        if len(self.layer_shape) == 1:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((3),(0)))# * prior_z_vals
        self.W_delta = np.tensordot(prior_activations, del_l, axes=0)
        self.B_delta = del_l

        #print("\nweights shape: ", self.weights.shape)
        #print("weights Delta shape: ", self.W_delta.shape)
        return(del_l_neg1, self.W_delta, self.B_delta)

class Conv_Block:
    def __init__(self, prev_bk_depth, num_filters, kernel_shape, stride, layer_shape, index):
        self.num_filters = num_filters
        self.kernel_shape = kernel_shape
        self.stride = stride
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_depth = prev_bk_depth


        self.b_type = "conv"
        self.ly_dim = len(layer_shape)
        self.filters = self.init_filters(self.num_filters, self.kernel_shape, init = True)
        self.feature_maps, self.feature_map_z_vals = self._init_feature_maps()
        self.feature_map_biases = np.zeros_like(self.feature_maps)
        self.biases = np.zeros(layer_shape) #idk if i need this but we can remove it later
        self.activations = self.feature_maps
        self.z_values = self.feature_map_z_vals
        self.submaps = np.zeros(shape=(self.prev_bk_depth, self.num_filters, *self.layer_shape))
        self.W_delta = np.zeros_like(self.feature_maps)        
        self.B_delta = np.zeros_like(self.feature_maps)#not sure about this

    def _init_feature_maps(self):
        feature_maps = np.zeros(shape=(self.num_filters, *self.layer_shape))
        feature_map_z_vals = np.zeros_like(feature_maps)  
        return(feature_maps, feature_map_z_vals)

    @staticmethod    
    def init_filters(num_filters:int, kernel_shape:tuple, init=True):
        if init:
            filters = np.random.uniform(-1,1, size=(num_filters, *kernel_shape))
        else:
            filters = np.zeros((num_filters, *kernel_shape))
        return (filters)
    
    def forward(self, input:np.ndarray):
        #input should be an ndarray of h num_channels i hieght j width
        for m in range(self.feature_maps.shape[0]):
            if len(input.shape) == 2:
                input = np.expand_dims(input, axis=0)
            temp_maps = np.zeros_like(input)
            for h in range(input.shape[0]):
                temp_maps[h, :, :] = convolve(input[h,:,:], self.filters[m,:,:], conv_mode="valid")
            self.submaps[:,m,:,:] = temp_maps
            self.feature_map_z_vals[m:,:] = np.sum(temp_maps, axis=0) + self.feature_map_biases[m, :,:]
        self.feature_maps = ReLU(self.feature_map_z_vals)
        self.activations = self.feature_maps
        self.z_values = self.feature_map_z_vals



    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if len(prior_activations.shape) == 2:
            prior_activations = np.expand_dims(prior_activations, axis=0)
            #might not need this code
        print("submaps shape: ", self.submaps.shape)
        error_anti_sum = np.zeros_like(self.submaps)
        print("error_anti_sum shape: ", error_anti_sum.shape)
        prime_anti_sum = ReLU_prime(self.submaps)
        neg1_error = np.zeros_like(prime_anti_sum)
        print("del_l shape", del_l.shape)

        w_delta_tmp =np.zeros(shape=(self.prev_bk_depth, *self.filters.shape))
        print("w_delta_tmp shape", w_delta_tmp.shape)
        w_delta = np.zeros_like(self.filters)

        print("z_vals shape: ", self.z_values.shape)
        print("prior_activations shape: ", prior_activations.shape)



        for m in range(self.submaps.shape[1]):
            for h in range(self.submaps.shape[0]):
                op1 =  (self.submaps[h,m,:,:] / self.z_values[m,:,:])
                op2 = del_l[m,:,:] * op1
                error_anti_sum[h,m,:,:] = op2

                w_delta_tmp[h,m,:,:] = fast_patch_convolve(prior_activations[h,:,:], self.kernel_shape,
                                                           error_anti_sum[h,m,:,:])
            w_delta[m,:,:] = np.sum(w_delta_tmp[:,m,:,:], axis=0)
        self.W_delta = w_delta


        for m in range(self.submaps.shape[1]):
            for h in range(self.submaps.shape[0]):
                #print("del_l slice shape, filter slice shape: ", del_l[m,:,:].shape, self.filters[m,:,:].shape)
                op1 = convolve(del_l[m,:,:], self.filters[m,:,:], conv_mode="valid")
                #print("op1 shape: ", op1.shape)
                #print("anti sum slice shape: ", anti_sum[h,m,:,:].shape)
                neg1_error[h, m, :, :] = op1 * prime_anti_sum[h,m,:,:]
                #neg1_error[h, m, :, :] = convolve(self.submaps[h,m,:,:], self.filters[m,:,:], conv_mode="valid") 

        del_l_neg1 = np.mean(neg1_error, axis=1)
        print("l-1 ERROR shape: ", del_l_neg1.shape)
        return(del_l_neg1, self.W_delta, self.B_delta)

class Pooling_ly:
    def __init__(self, shape, num_filters, index, stride:int=2, p_type:str="max"):
        self.index = index
        self.shape = shape
        self.depth = num_filters
        self.stride = stride
        self.p_type = p_type
        self.b_type = "pooling"


        self.activations = np.ndarray(shape=(self.depth, *self.shape))
        self.z_values = np.zeros_like(self.activations)

    def pooling(self, input_act):
        if len(input_act.shape) == 2:
            input_act = np.expand_dims(input_act, axis=0)
        
        self.activations = np.zeros((input_act.shape[0], *self.shape))
        for i in range(input_act.shape[0]):
            self.activations[i,:,:] = pool(input_act[i,:,:], stride=self.stride, mode=self.p_type)
        

    def forward(self, input):
        self.pooling(input)
        self.z_values = self.activations

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        #print("pooling del_l input shape: ", del_l.shape)
        if expected is not None:
            del_l = (expected - self.activations) * (self.activations) #Suspect

        #print("current layer shape: ", self.shape)
        #print("del_l shape: ", del_l.shape)
        
        del_l_neg1 = np.repeat(np.repeat(del_l, 2, axis=1), 2, axis=2) #should be expanded over axes 1,2. axis 0 is the feature map dim
        #print("pooling del_l next layer back shape:", del_l_neg1.shape)
        #MIGHT NEED TO CORRECT THIS TO ONLY PROPIGATE ERROR TO THE LOCATIONS WHICH TRIGGERED MAX POOL
        self.W_delta = np.zeros_like(self.activations)        
        self.B_delta = np.zeros_like(self.activations)

        return(del_l_neg1, self.W_delta, self.B_delta)

class NN:
    def __init__(self, config:DictConfig, blocks: list):
        self.config = config
        self.blocks = blocks

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):
        index_to_block = build_index_lookup(cfg)
        print("creating network....")

        blocks = []
        for items in cfg.blocks:
            #print(items)
            block_name, block_data = index_to_block[cfg.blocks[items].index]
            

            if block_data.type == "input":
                blocks.append(Input_block(block_data.shape, block_data.index))
            if block_data.type == "fc":
                if prev_bk_data.type == "conv2D":
                    prev_bk_shp = (prev_bk_data.filters.filter_num, *prev_bk_data.shape)
                    blocks.append(FC_CONV_block(prev_bk_shp, block_data.shape, block_data.index))
                else:
                    blocks.append(FC_block(prev_bk_data.shape, block_data.shape, block_data.index))
            if block_data.type == "pool":
                blocks.append(Pooling_ly(block_data.shape, prev_bk_data.filters.filter_num, block_data.index, block_data.stride, block_data.mode))

            if block_data.type == "conv2D":
                fltr = block_data.filters
                if prev_bk_data.type == "conv2D":
                    prev_bk_dpth = prev_bk_data.filters.filter_num
                elif prev_bk_data.type == "pool":
                    prev_bk_dpth = blocks[-1].depth
                else:
                    prev_bk_dpth = 1

                blocks.append(Conv_Block(prev_bk_dpth, fltr.filter_num, fltr.kernel_shape, fltr.stride, block_data.shape, block_data.index))

            prev_bk_name = block_name
            prev_bk_data = block_data

        print("Done!")
        return cls(cfg, blocks)
    
    def forward(self, input_ly):
        print("running forward function....")
        self.blocks[0].activations = input_ly
        for index in range(len(self.blocks)):
            #print("\nindex: ", index)
            if index == 0:
                self.blocks[0].activations = input_ly
            else:
                self.blocks[index].forward(self.blocks[index-1].activations)
        print("Done!\n")
            

    def backprop(self, expected, lr):
        print("running backprop....")
        dW = []
        dB = []
        for index in range(len(self.blocks)-1, 0 , -1):


            print("index: ", index)
            if index == len(self.blocks)-1:
                delL_back1, dW_tmp, dB_tmp = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         expected = expected
                                                         )
            else:
                delL_back1, dW_tmp, dB_tmp = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         del_l = delL_back1
                                                         )
            dW.append(dW_tmp)
            dB.append(dB_tmp)
        dW.reverse()
        dB.reverse()#doing this because they were created from back to front
        print("Done!")
        return(dW, dB)

def batch_backprop(trn_cfg:DictConfig, mdl_cfg:DictConfig, n_network):

    return(mean_dW, mean_dB, mean_loss)

def train(trn_cfg:DictConfig, mdl_cfg:DictConfig):
    """
    this is the core training loop that trains through epochs.  
    requires training_cfg(trn_cfg) containing hyperparameters and image and label paths,
    and model_cfg(mdl_cfg) containing model architecture.

    """


    for epochs in range(trn_cfg.epochs):

        for batches in range(trn_cfg.batch_size):
            batch_dW = []
            batch_dB = []
            print(".")

            #create batches of batch size, feed forward, backprop, save dW, dB temporayly, avereage accross batch,
            #apply averaged dW, dB to network, repeat for all batchs
        #repeat for all epochs

    
    return(0)

#todo:
    #figure out network autocreation DONE!
    #figure out padding algorrithm DONE!
    #figure out down sizing 
        #this will be through pooling
    #write forward functions DONE!
        #since these differ between different types of layers and blocks, 
        # maybe each block should have a forward function?
    #write backprop
        #now done with symbolic backprop creation, need to consider:
        #should there be a backprop function for each block/layer or a golbal
    #revaluate model architechure for practical ability to detect objects

    #make a visualizer
    



In [6]:
mdl_cfg = OmegaConf.load("model.yaml")
lrn_cfg = OmegaConf.load("hyper_params.yaml")

model = NN.create_network(mdl_cfg)

input_ly = np.random.rand(28,28)
model.forward(input_ly)
expected  = np.zeros((10,))
expected[3] = 1

mdl = model.blocks
#for blocks in mdl:
#    if blocks.b_type == "conv":
#        print("SUBMAPS shape: ",blocks.submaps.shape)

model.backprop(expected, 0.05)

train(lrn_cfg, mdl_cfg)

creating network....
Done!
running forward function....
Done!

running backprop....
index:  7
index:  6
index:  5
index:  4
submaps shape:  (27, 9, 14, 14)
error_anti_sum shape:  (27, 9, 14, 14)
del_l shape (9, 14, 14)
w_delta_tmp shape (27, 9, 3, 3)
z_vals shape:  (9, 14, 14)
prior_activations shape:  (27, 14, 14)
l-1 ERROR shape:  (27, 14, 14)
index:  3
submaps shape:  (21, 27, 14, 14)
error_anti_sum shape:  (21, 27, 14, 14)
del_l shape (27, 14, 14)
w_delta_tmp shape (21, 27, 3, 3)
z_vals shape:  (27, 14, 14)
prior_activations shape:  (21, 14, 14)
l-1 ERROR shape:  (21, 14, 14)
index:  2
index:  1
submaps shape:  (1, 21, 28, 28)
error_anti_sum shape:  (1, 21, 28, 28)
del_l shape (21, 28, 28)
w_delta_tmp shape (1, 21, 3, 3)
z_vals shape:  (21, 28, 28)
prior_activations shape:  (1, 28, 28)
l-1 ERROR shape:  (1, 28, 28)
Done!
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.

0

In [4]:
print(model.blocks[4].W_delta.shape)
print(model.blocks[4].W_delta)


(9, 3, 3)
[[[-2.19965936e+13 -2.05564435e+13 -1.85118719e+13]
  [-1.88010674e+13 -1.79284590e+13 -1.62761270e+13]
  [-1.56593428e+13 -1.46074885e+13 -1.43329806e+13]]

 [[-5.10252126e+12 -3.47553488e+12 -2.83561389e+11]
  [-7.41571101e+12 -6.04602879e+12 -3.18607691e+12]
  [-8.69332081e+12 -6.78333721e+12 -4.05042591e+12]]

 [[-1.38507009e+12 -1.51735172e+12  3.21365026e+12]
  [-8.91436854e+11 -3.60705414e+12 -8.72173512e+11]
  [-2.81905022e+12 -2.68691317e+12 -3.08895337e+11]]

 [[-1.54930389e+13 -1.49145145e+13 -1.73635258e+13]
  [-1.55591228e+13 -1.37546476e+13 -1.44727560e+13]
  [-1.32482161e+13 -1.22150708e+13 -1.19958744e+13]]

 [[ 9.46747631e+12  7.99092826e+12  7.48129822e+12]
  [ 6.51067264e+12  5.28812607e+12  5.56527287e+12]
  [ 7.20756301e+12  6.04815138e+12  6.25853911e+12]]

 [[ 4.46823598e+13  2.14385646e+13  1.62374690e+13]
  [-4.02914992e+13 -7.56321423e+13 -8.20876667e+13]
  [ 7.84697752e+12 -1.26835559e+13 -2.54954562e+13]]

 [[-4.34079239e+12 -5.73602210e+12 -4.8292

In [ ]:
item = model.blocks
print("blocks")
for items in item:
    print(items.activations.shape, items.index, items.b_type)
    name = f"block_{items.index}_Activations"
    globals()[name] = items.activations




In [ ]:
mdl_cfg = OmegaConf.load("model.yaml")
print(type(mdl_cfg))
print(mdl_cfg)
examine_1 = mdl_cfg.model.layers.input_ly.shape
print("\nexamine_1: ")
print(examine_1)
print(type(examine_1))
examine_2 = tuple(mdl_cfg.model.layers.input_ly.shape)
print("\nexamine_2: ")
print(examine_2)
print(type(examine_2))
#next to figure out how to handle .yaml files and DictConfig files

#net = NN.create_network(cfg)


In [ ]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)